# Unscented Kalman Filter — Experimentation

## Overview

This notebook demonstrates the UKF on a **CTRV target with a non-differentiable radar measurement model**: a sensor fixed at the origin measures the **integrated optical path length** and **bearing** to a manoeuvring target. The measurement function has no closed-form Jacobian, making the EKF's linearisation approach intractable in principle and $\varepsilon$-sensitive in practice.

### Why this scenario requires the UKF

The radar does not measure clean geometric range. The signal travels through an atmosphere whose refractive index varies along the path, so the actual observed range is:

$$
r_\text{obs} = \int_0^{r_\text{geom}} n(s,\, \mathbf{x})\, ds
$$

Applying Leibniz's rule to differentiate this integral w.r.t. $\mathbf{x}$ yields another integral — there is **no analytical Jacobian**. The EKF must fall back on numerical finite differences, incurring $2n = 10$ extra evaluations of $h$ per timestep and introducing an $\varepsilon$-tuning problem with no principled stopping criterion.

The UKF propagates $2n+1 = 11$ sigma points through $h$ directly — the same numerical integral, called 11 times — requiring no Jacobian at all. No $\varepsilon$, no approximation beyond the unscented transform itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import chi2
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

np.random.seed(42)

## Mathematical Setup

### State and Process Model (CTRV)

The state vector is $\mathbf{x} = [p_x,\; p_y,\; v,\; \theta,\; \omega]^\top$, following the **Constant Turn Rate and Velocity** (CTRV) model:

$$
f(\mathbf{x}, \Delta t) = \begin{bmatrix}
p_x + \frac{v}{\omega}(\sin(\theta + \omega\Delta t) - \sin\theta) \\
p_y + \frac{v}{\omega}(-\cos(\theta + \omega\Delta t) + \cos\theta) \\
v \\ \theta + \omega\Delta t \\ \omega
\end{bmatrix}
$$

This is **nonlinear** in both $\theta$ and $\omega$. A standard KF cannot be applied; the EKF requires a 5×5 Jacobian $\mathbf{F}_J$ with trigonometric and rational entries. The UKF propagates sigma points through $f$ directly — no Jacobian required.

### Measurement Model (Non-Differentiable Radar)

$$
\mathbf{h}(\mathbf{x}) = \begin{bmatrix} r_\text{geom} + \int_0^{r_\text{geom}} 315\times10^{-6} e^{-s\cdot0.05/7350}\left(1 + 0.3\sin^2\!\tfrac{\pi s}{r}\right) ds \\[4pt] \arctan2(p_y, p_x) \end{bmatrix}
$$

The integral has no closed form, so $\partial \mathbf{h}/\partial \mathbf{x}$ requires another integral — no analytical Jacobian exists.

### UKF: Merwe Scaled Sigma Points

For $n=5$, eleven sigma points are drawn from $(\hat{\mathbf{x}}, \mathbf{P})$ and propagated through $f$ and $h$ exactly:

$$
\mathcal{X}_0 = \hat{\mathbf{x}}, \quad
\mathcal{X}_i = \hat{\mathbf{x}} + \left(\sqrt{(n+\lambda)\mathbf{P}}\right)_{:,i}, \quad
\mathcal{X}_{n+i} = \hat{\mathbf{x}} - \left(\sqrt{(n+\lambda)\mathbf{P}}\right)_{:,i}
$$

with $\lambda = \alpha^2(n+\kappa) - n$, $\alpha=10^{-3}$, $\beta=2$, $\kappa=0$.

In [ ]:
# ============================================================
#  CTRV process model
# ============================================================

def ctrv_f(x, dt):
    """Nonlinear CTRV process. State: [px, py, v, theta, omega]."""
    px, py, v, theta, omega = x
    if abs(omega) < 1e-6:
        return np.array([px + v*np.cos(theta)*dt,
                         py + v*np.sin(theta)*dt,
                         v, theta, omega])
    t_new = theta + omega * dt
    return np.array([
        px + (v / omega) * (np.sin(t_new) - np.sin(theta)),
        py + (v / omega) * (-np.cos(t_new) + np.cos(theta)),
        v, t_new, omega
    ])


def simulate_ctrv(x0, N, dt):
    states = np.zeros((N, 5))
    states[0] = x0
    for k in range(N - 1):
        states[k + 1] = ctrv_f(states[k], dt)
    return states


# ============================================================
#  Non-differentiable radar measurement model
# ============================================================

def h_radar(x):
    """
    Radar measurement: [integrated optical path length, bearing].
    The sin^2(pi*s/r) term inside the integrand couples s and r,
    so d/dx[integral] requires another integral — no closed form.
    No analytical Jacobian exists.
    """
    px, py = x[0], x[1]
    r = np.hypot(px, py)
    bearing = np.arctan2(py, px)

    def refractivity(s):
        return 315e-6 * np.exp(-s * 0.05 / 7350) * (1 + 0.3 * np.sin(np.pi * s / r)**2)

    delta_r, _ = quad(refractivity, 0, r)
    return np.array([r + delta_r, bearing])


def simulate_radar_measurements(states, sigma_r, sigma_theta):
    N = len(states)
    z = np.zeros((N, 2))
    for k in range(N):
        z_clean = h_radar(states[k])
        z[k, 0] = z_clean[0] + np.random.randn() * sigma_r
        z[k, 1] = z_clean[1] + np.random.randn() * sigma_theta
    return z


# ============================================================
#  UKF core
# ============================================================

def ukf_weights(n, alpha=1e-3, beta=2.0, kappa=0.0):
    lam  = alpha**2 * (n + kappa) - n
    Wm   = np.full(2*n + 1, 0.5 / (n + lam))
    Wc   = Wm.copy()
    Wm[0] = lam / (n + lam)
    Wc[0] = lam / (n + lam) + (1 - alpha**2 + beta)
    return Wm, Wc, lam


def ukf_sigma_points(x, P, lam):
    n    = len(x)
    L    = np.linalg.cholesky((n + lam) * P)
    sig  = np.empty((2*n + 1, n))
    sig[0] = x
    for i in range(n):
        sig[i + 1]     = x + L[:, i]
        sig[n + i + 1] = x - L[:, i]
    return sig


def ukf_predict(x, P, Q, dt, Wm, Wc, lam):
    sig   = ukf_sigma_points(x, P, lam)
    sig_f = np.array([ctrv_f(s, dt) for s in sig])
    x_pred = Wm @ sig_f
    diffs  = sig_f - x_pred
    P_pred = Q + sum(Wc[i] * np.outer(diffs[i], diffs[i]) for i in range(len(Wm)))
    return x_pred, P_pred, sig_f


def ukf_update(x_pred, P_pred, sig_f, z, R, Wm, Wc):
    """UKF update: h_radar evaluated at sigma points — no Jacobian."""
    sig_h  = np.array([h_radar(s) for s in sig_f])
    z_pred = Wm @ sig_h
    dz     = sig_h - z_pred
    dx     = sig_f - x_pred
    S    = R + sum(Wc[i] * np.outer(dz[i], dz[i]) for i in range(len(Wm)))
    Pxz  = sum(Wc[i] * np.outer(dx[i], dz[i])     for i in range(len(Wm)))
    K    = Pxz @ np.linalg.inv(S)
    innov    = z - z_pred
    innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
    x_upd  = x_pred + K @ innov
    P_upd  = 0.5 * (P_pred - K @ S @ K.T)
    P_upd  = P_upd + P_upd.T
    nis    = float(innov @ np.linalg.inv(S) @ innov)
    return x_upd, P_upd, nis

---
## Scenario — CTRV Target with Non-Differentiable Radar Measurement

A target follows a gentle curved arc (constant turn rate) while a fixed sensor measures integrated optical path length and bearing. Both the process model and measurement model are nonlinear. A standard KF cannot be applied to either.

**Parameters:**

| | Value | Note |
|---|---|---|
| $\mathbf{x}_0^\text{true}$ | $[600, -200, 5, 150°, 0.025]^\top$ | Starts east, curves northward |
| $\Delta t$ | $1.0$ s | |
| $N$ | $100$ steps | |
| $\sigma_r$ | $20$ m | Optical path length noise |
| $\sigma_\theta$ | $2°$ | Bearing noise |
| $\mathbf{P}_0$ | $\text{diag}(\sigma_r^2, \sigma_r^2, 5^2, 30°^2, 3°^2)$ | Initialised from first measurement |

In [ ]:
# ============================================================
#  Scenario setup and UKF run
# ============================================================

dt       = 1.0
N        = 100
sigma_r  = 20.0
sigma_theta = np.deg2rad(2.0)

# True trajectory
x0_true = np.array([600.0, -200.0, 5.0, np.deg2rad(150.0), 0.025])
true_states = simulate_ctrv(x0_true, N, dt)

R = np.diag([sigma_r**2, sigma_theta**2])
z_meas = simulate_radar_measurements(true_states, sigma_r, sigma_theta)

Q = np.diag([0.1**2, 0.1**2, 0.3**2,
             np.deg2rad(1.)**2, np.deg2rad(0.5)**2])

# Initialise from first measurement
r0, th0 = z_meas[0]
x0_ukf = np.array([r0*np.cos(th0), r0*np.sin(th0),
                   5.0, np.deg2rad(150.0), 0.0])
P0 = np.diag([sigma_r**2, sigma_r**2, 5.**2,
              np.deg2rad(30.)**2, np.deg2rad(3.)**2])

Wm, Wc, lam = ukf_weights(5)

# Run UKF
estimates = np.zeros((N, 5))
P_all     = []
nis_all   = np.zeros(N)

x, P = x0_ukf.copy(), P0.copy()
for k in range(N):
    x, P, sig_f = ukf_predict(x, P, Q, dt, Wm, Wc, lam)
    x, P, nis   = ukf_update(x, P, sig_f, z_meas[k], R, Wm, Wc)
    estimates[k] = x
    P_all.append(P.copy())
    nis_all[k]   = nis

# Metrics
pos_err = np.sqrt(
    (estimates[:, 0] - true_states[:, 0])**2 +
    (estimates[:, 1] - true_states[:, 1])**2
)
rmse     = np.sqrt(np.mean(pos_err**2))
mean_nis = np.mean(nis_all)

NIS_LOWER = chi2.ppf(0.025, df=2)
NIS_UPPER = chi2.ppf(0.975, df=2)
pct_in    = np.mean((nis_all >= NIS_LOWER) & (nis_all <= NIS_UPPER)) * 100

print(f'Position RMSE : {rmse:.1f} m')
print(f'Mean NIS      : {mean_nis:.3f}  (expected 2.000 for chi²(2))')
print(f'NIS in bounds : {pct_in:.0f}%  (expected ~95%)')

In [ ]:
# ============================================================
#  Plot: Trajectory + Position Error
# ============================================================

def draw_cov_ellipse(ax, center, P_pos, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(P_pos)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle  = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width  = 2 * n_std * np.sqrt(max(vals[0], 0))
    height = 2 * n_std * np.sqrt(max(vals[1], 0))
    el = Ellipse(xy=center, width=width, height=height, angle=angle, **kwargs)
    return ax.add_patch(el)


fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── LEFT: Trajectory ─────────────────────────────────────────────────────────
ax = axes[0]

pts  = np.array([true_states[:, 0], true_states[:, 1]]).T.reshape(-1, 1, 2)
segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
lc   = LineCollection(segs, cmap='plasma', norm=plt.Normalize(0, N),
                      linewidth=2.5, zorder=6)
lc.set_array(np.arange(N - 1))
ax.add_collection(lc)
cbar = fig.colorbar(lc, ax=ax, pad=0.02)
cbar.set_label('Time step', fontsize=16)
cbar.ax.tick_params(labelsize=15)

ax.plot(estimates[:, 0], estimates[:, 1],
        'k--', linewidth=1.5, alpha=0.7, zorder=5)
ax.scatter([0], [0], s=300, c='black', marker='^', zorder=8)

# Covariance ellipses at t=0, t=40, t=99
for t_idx, color, al in [(0, '#d62728', 0.28), (40, '#ff7f0e', 0.28), (99, '#1f77b4', 0.35)]:
    P_pos_k = P_all[t_idx][np.ix_([0, 1], [0, 1])]
    center  = (estimates[t_idx, 0], estimates[t_idx, 1])
    draw_cov_ellipse(ax, center, P_pos_k, n_std=2,
                     facecolor=color, edgecolor=color, alpha=al,
                     linewidth=1.5, zorder=3, clip_on=True)

ax.set_xlabel('$p_x$ (m)', fontsize=17)
ax.set_ylabel('$p_y$ (m)', fontsize=17)
ax.set_title('Trajectory', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.4)

legend_handles = [
    Line2D([0], [0], color='darkorchid', linewidth=2.5),
    Line2D([0], [0], color='k', linewidth=1.5, linestyle='--', alpha=0.7),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='black',
           markersize=11, linestyle='None'),
]
ax.legend(legend_handles, ['True position', 'UKF estimate', 'Sensor'],
          fontsize=14, loc='upper left')

# ── RIGHT: Position error + noise floor + RMSE ───────────────────────────────
ax = axes[1]

r_true      = np.hypot(true_states[:, 0], true_states[:, 1])
noise_floor = np.sqrt(sigma_r**2 + (r_true * sigma_theta)**2)

ax.plot(pos_err, color='steelblue', linewidth=1.8,
        label='UKF position error (m)')
ax.plot(noise_floor, color='#d62728', linewidth=1.5, linestyle=':',
        label=r'Single-measurement noise floor' '\n'
              r'$\sqrt{\sigma_r^2 + (r\sigma_\theta)^2}$'
              f' (mean {noise_floor.mean():.1f} m)')
ax.axhline(rmse, color='k', linewidth=1.5, linestyle='--',
           label=f'RMSE = {rmse:.1f} m')

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('Position error (m)', fontsize=17)
ax.set_title('Position Error vs Measurement Noise Floor', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.set_ylim(bottom=0)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.4)

plt.suptitle('UKF: CTRV with Non-Differentiable Radar Measurement',
             fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Measurements Plot
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

t = np.arange(N)

# ── LEFT: Optical path length measurements vs true range ─────────────────────
ax = axes[0]

r_true = np.hypot(true_states[:, 0], true_states[:, 1])
r_true_obs = np.array([h_radar(true_states[k])[0] for k in range(N)])  # true optical path

sc = ax.scatter(t, z_meas[:, 0], c=t, cmap='plasma', s=18, zorder=4,
                label='Measured optical path length')
ax.plot(r_true_obs, color='k', linewidth=1.5, linestyle='--', zorder=3,
        label='True optical path length')
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Time step', fontsize=16)
cbar.ax.tick_params(labelsize=15)

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('Range (m)', fontsize=17)
ax.set_title('Optical Path Length Measurements', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.4)

# ── RIGHT: Bearing measurements vs true bearing ───────────────────────────────
ax = axes[1]

bearing_true = np.arctan2(true_states[:, 1], true_states[:, 0])

sc2 = ax.scatter(t, np.rad2deg(z_meas[:, 1]), c=t, cmap='plasma', s=18, zorder=4,
                 label='Measured bearing')
ax.plot(np.rad2deg(bearing_true), color='k', linewidth=1.5, linestyle='--', zorder=3,
        label='True bearing')
cbar2 = fig.colorbar(sc2, ax=ax, pad=0.02)
cbar2.set_label('Time step', fontsize=16)
cbar2.ax.tick_params(labelsize=15)

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('Bearing (°)', fontsize=17)
ax.set_title('Bearing Measurements', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.4)

plt.suptitle('UKF: Raw Measurements', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  NIS Plot
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(nis_all, color='steelblue', linewidth=1.5, alpha=0.85,
        label='NIS $\\varepsilon_k$')
ax.axhline(NIS_UPPER, color='red', linestyle='--', linewidth=1.5,
           label=f'$\\chi^2_2$ 97.5% = {NIS_UPPER:.2f}')
ax.axhline(NIS_LOWER, color='orange', linestyle='--', linewidth=1.5,
           label=f'$\\chi^2_2$ 2.5% = {NIS_LOWER:.3f}')
ax.axhline(2.0, color='grey', linestyle=':', linewidth=1.2,
           label='Expected mean = 2')
ax.fill_between(range(N), NIS_LOWER, NIS_UPPER,
                color='green', alpha=0.07, label='95% consistency band')

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('NIS', fontsize=17)
ax.set_title(
    f'UKF: Normalised Innovation Squared\n'
    f'Mean NIS = {mean_nis:.3f}   |   '
    f'{pct_in:.0f}% within 95% $\\chi^2_2$ bounds (consistent filter expects ~95%)',
    fontsize=15
)
ax.tick_params(axis='both', labelsize=15)
ax.set_ylim(bottom=0)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Analysis

**Why the UKF succeeds here.** The CTRV process model and the atmospheric radar measurement are both highly nonlinear. The UKF propagates eleven sigma points through each function directly — no Jacobian is computed at any stage. The unscented transform captures the true shape of the propagated distribution to second order, making the predicted innovation covariance $\mathbf{S}_k$ an accurate reflection of actual filter uncertainty.

**NIS consistency.** The NIS scatters around the expected mean of 2 (for a 2D measurement, $\chi^2_2$) with approximately 95% of values within the bounds. This confirms the filter is well-calibrated — it is neither overconfident (NIS persistently above the upper bound) nor underconfident (NIS persistently near zero).

**RMSE below the noise floor.** The single-measurement noise floor $\sqrt{\sigma_r^2 + (r\sigma_\theta)^2}$ represents the position error achievable from a single range-bearing observation. The filter's RMSE falls below this floor because it **fuses multiple measurements over time**, accumulating information and averaging out noise. The Kalman gain weights each measurement by the relative size of process and measurement uncertainty, yielding estimates more accurate than any single observation.

**Covariance ellipses.** The uncertainty ellipses on the trajectory panel shrink from the initial (red) to the final (blue) timestep, reflecting genuine information gain. The orientation of each ellipse reflects the direction of remaining ambiguity — typically aligned with the radial direction early on, where range uncertainty dominates.